# Chapter 07 — Model Training & Optimization

Chapter 06에서 만든 신경망을 안정적으로 학습시키는 방법을 실습합니다. 이번 Notebook은 **추가 CSV 파일 없이 CPU에서 실행**할 수 있도록 작은 합성 데이터셋을 내부에서 생성합니다. 실제 프로젝트의 요구사항과 전체 데이터는 Chapter 07의 `README.md` 및 `projects/`에서 확인하세요.

## 1. 실행 환경 확인

PyTorch, NumPy, scikit-learn이 설치되어 있다고 가정합니다. Colab에는 일반적으로 기본 제공되지만 환경에 따라 버전이 달라질 수 있습니다.

In [ ]:
import copy
import random
import tempfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(1)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. 데이터 생성 및 분할

입력값 `x1, x2`로부터 비선형 Target을 생성합니다. Train·Validation·Test를 먼저 나누고, **Train의 통계량만**으로 Scaling합니다.

In [ ]:
rng = np.random.default_rng(42)
raw_x = rng.normal(size=(240, 2)).astype(np.float32)
raw_y = (0.5 + raw_x[:,0]**2 - 0.6*raw_x[:,1] +
         0.15*np.sin(3*raw_x[:,1]) + rng.normal(0,0.12,240)).astype(np.float32)

split = (slice(0,150), slice(150,195), slice(195,240))
train_raw = raw_x[split[0]]
mean = train_raw.mean(axis=0)
std = np.where(train_raw.std(axis=0) < 1e-8,1.,train_raw.std(axis=0))
y_mean = float(raw_y[split[0]].mean())
y_std = float(raw_y[split[0]].std())
x_all = torch.from_numpy((raw_x - mean)/std)
y_all = torch.from_numpy(((raw_y-y_mean)/y_std).reshape(-1,1))
(x_train,y_train),(x_val,y_val),(x_test,y_test) = [
    (x_all[s],y_all[s]) for s in split]
print("split sizes:",len(x_train),len(x_val),len(x_test))

## 3. 신경망과 Mini-Batch

`nn.Linear(2,16) → ReLU → nn.Linear(16,1)`을 사용합니다. Batch Size는 24로 설정하고 마지막 Batch도 포함합니다.

In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,1))

def make_train_loader(epoch=0, batch_size=24):
    generator = torch.Generator().manual_seed(100+epoch)
    return DataLoader(TensorDataset(x_train,y_train),batch_size=batch_size,
                      shuffle=True,generator=generator)

model=make_model()
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("Output Shape:",tuple(model(x_train[:3]).shape))

## 4. Training Loop

각 Batch에서 Gradient를 초기화한 뒤 Forward → Loss → Backward → Step을 실행합니다. 평균 Loss는 각 Batch의 Sample 수로 가중합니다.

In [ ]:
def train_epoch(model,optimizer,loader):
    model.train()
    total=0.0;size=0
    for x,y in loader:
        optimizer.zero_grad(set_to_none=True)
        loss=nn.MSELoss()(model(x),y)
        loss.backward()
        optimizer.step()
        total+=float(loss.item())*len(x)
        size+=len(x)
    return total/size

def valid_loss(model):
    model.eval()
    with torch.inference_mode():
        return float(nn.MSELoss()(model(x_val),y_val).item())

optimizer=torch.optim.AdamW(model.parameters(),lr=0.02,weight_decay=0.01)
records=[]
for epoch in range(1,11):
    train_value=train_epoch(model,optimizer,make_train_loader(epoch))
    val_value=valid_loss(model)
    records.append((epoch,train_value,val_value))
print("first/last:",records[0],records[-1])

## 5. SGD와 AdamW 비교

공정한 비교를 위해 두 모델은 동일한 초기 Weight를 사용합니다. 마지막 Epoch가 아닌 Validation 결과도 함께 기록합니다.

In [ ]:
torch.manual_seed(2026)
initial=copy.deepcopy(make_model().state_dict())
comparison={}
for name in ("sgd_momentum","adamw"):
    net=make_model();net.load_state_dict(initial)
    opt=(torch.optim.SGD(net.parameters(),lr=.04,momentum=.85)
         if name=="sgd_momentum" else torch.optim.AdamW(net.parameters(),lr=.015))
    track=[]
    for epoch in range(1,13):
        train_epoch(net,opt,make_train_loader(epoch))
        net.eval()
        with torch.inference_mode():
            track.append(float(nn.MSELoss()(net(x_val),y_val)))
    comparison[name]=track
print({k:round(min(v),4) for k,v in comparison.items()})

## 6. Learning Rate Scheduler

StepLR는 Epoch가 끝난 뒤 `scheduler.step()`을 호출합니다. 이번 Epoch에 실제 사용한 LR은 호출 전에 기록합니다.

In [ ]:
net=make_model()
opt=torch.optim.SGD(net.parameters(),lr=.1)
sched=torch.optim.lr_scheduler.StepLR(opt,step_size=2,gamma=.5)
lr_history=[]
for epoch in range(1,7):
    lr_history.append(round(opt.param_groups[0]['lr'],5))
    train_epoch(net,opt,make_train_loader(epoch))
    sched.step()
print("LR history:",lr_history)
assert lr_history==[.1,.1,.05,.05,.025,.025]

## 7. Early Stopping

Validation Loss가 개선된 Epoch의 State를 별도로 복사해 저장합니다. Patience를 넘으면 종료하고 Best State를 복원합니다.

In [ ]:
torch.manual_seed(18)
net=make_model();opt=torch.optim.AdamW(net.parameters(),lr=.02)
best=float('inf');best_state=None;best_epoch=0;stale=0
for epoch in range(1,41):
    train_epoch(net,opt,make_train_loader(epoch))
    net.eval()
    with torch.inference_mode():
        v=float(nn.MSELoss()(net(x_val),y_val))
    if v < best-1e-6:
        best=v;best_state=copy.deepcopy(net.state_dict());best_epoch=epoch;stale=0
    else:
        stale+=1
        if stale>=6:break
net.load_state_dict(best_state)
print("best_epoch:",best_epoch,"stopped_at:",epoch,"best_val_loss:",round(best,4))

## 8. Dropout과 평가 모드

Dropout은 Train과 Evaluation에서 다른 동작을 합니다. `model.eval()`은 Gradient 비활성화와 별도입니다.

In [ ]:
drop_model=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Dropout(.5),nn.Linear(8,1))
example=x_val[:6]
drop_model.train()
train_a=drop_model(example);train_b=drop_model(example)
drop_model.eval()
with torch.inference_mode():
    eval_a=drop_model(example);eval_b=drop_model(example)
print("Train equal:",torch.equal(train_a,train_b))
print("Eval equal:",torch.equal(eval_a,eval_b))

## 9. Loss 집계 검증

Batch Loss를 단순 평균하면 마지막 Batch의 비중이 커질 수 있습니다.

In [ ]:
batch_loss=[0.4,0.9]
batch_size=[8,2]
weighted=sum(l*n for l,n in zip(batch_loss,batch_size))/sum(batch_size)
naive=sum(batch_loss)/len(batch_loss)
print("weighted:",weighted,"naive:",naive)
assert abs(weighted-.5)<1e-9

## 10. Gradient Clipping

Clipping 전 Norm은 `clip_grad_norm_()`의 반환값으로 확인할 수 있습니다.

In [ ]:
net=nn.Linear(2,1)
opt=torch.optim.SGD(net.parameters(),lr=.01)
large_x=torch.full((8,2),100.)
opt.zero_grad(set_to_none=True)
loss=nn.MSELoss()(net(large_x),torch.zeros(8,1))
loss.backward()
before=float(nn.utils.clip_grad_norm_(net.parameters(),max_norm=.5))
after=float(torch.linalg.vector_norm(torch.cat([p.grad.flatten() for p in net.parameters()])))
print("before:",round(before,3),"after:",round(after,3))
assert after<=.50001

## 11. Gradient Accumulation

11개 Sample을 4·4·3개로 나누고 각 Micro-Batch Loss에 `크기/11`을 곱합니다. 이 예제에서는 Dropout·BatchNorm이 없는 Linear 모델을 사용합니다.

In [ ]:
torch.manual_seed(8)
x=torch.randn(11,2);y=torch.randn(11,1)
initial=copy.deepcopy(nn.Linear(2,1).state_dict())
full=nn.Linear(2,1);full.load_state_dict(initial)
full.zero_grad(set_to_none=True)
nn.MSELoss()(full(x),y).backward()
full_grad=torch.cat([p.grad.flatten() for p in full.parameters()])
micro=nn.Linear(2,1);micro.load_state_dict(initial)
micro.zero_grad(set_to_none=True)
for start in range(0,11,4):
    xb=x[start:start+4];yb=y[start:start+4]
    (nn.MSELoss()(micro(xb),yb)*len(xb)/11).backward()
micro_grad=torch.cat([p.grad.flatten() for p in micro.parameters()])
print("maximum gradient difference:",float((full_grad-micro_grad).abs().max()))
assert torch.allclose(full_grad,micro_grad,atol=1e-6)

## 12. Checkpoint 저장과 복원

학습 재개에는 Optimizer와 Scheduler State도 필요합니다. 이 Notebook은 자신이 만든 임시 Checkpoint만 불러옵니다.

In [ ]:
model=nn.Linear(2,1)
opt=torch.optim.AdamW(model.parameters(),lr=.02)
sched=torch.optim.lr_scheduler.StepLR(opt,step_size=2,gamma=.5)
for _ in range(3):
    opt.zero_grad(set_to_none=True)
    nn.MSELoss()(model(x_train[:12]),y_train[:12]).backward()
    opt.step();sched.step()
with tempfile.TemporaryDirectory() as temp:
    path=Path(temp)/'state.pt'
    torch.save({'epoch':3,'model':model.state_dict(),'optimizer':opt.state_dict(),
                'scheduler':sched.state_dict()},path)
    saved=torch.load(path,weights_only=True,map_location='cpu')
    restored=nn.Linear(2,1);new_opt=torch.optim.AdamW(restored.parameters(),lr=.02)
    new_sched=torch.optim.lr_scheduler.StepLR(new_opt,step_size=2,gamma=.5)
    restored.load_state_dict(saved['model'])
    new_opt.load_state_dict(saved['optimizer'])
    new_sched.load_state_dict(saved['scheduler'])
    with torch.inference_mode():
        diff=float((model(x_val)-restored(x_val)).abs().max())
print("Restored epoch:",saved['epoch'],"max prediction difference:",diff)
assert diff<1e-7

## 13. 최종 Test 평가

Validation 단계에서 선정한 Weight를 사용하며, Test 결과를 보고 학습 설정을 다시 바꾸지 않습니다.

In [ ]:
net.eval()
with torch.inference_mode():
    predicted=net(x_test).numpy().reshape(-1)*y_std+y_mean
    truth=y_test.numpy().reshape(-1)*y_std+y_mean
mae=float(np.mean(np.abs(predicted-truth)))
rmse=float(np.sqrt(np.mean((predicted-truth)**2)))
print(f'Test MAE={mae:.4f}, RMSE={rmse:.4f}')

## 14. ReduceLROnPlateau

StepLR와 달리 Validation 지표를 인자로 전달합니다. 실제 학습에 적용할 때는 `validation_loss()`의 반환값을 사용합니다.

In [ ]:
param=nn.Parameter(torch.tensor(1.))
opt=torch.optim.SGD([param],lr=.1)
plateau=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=1)
for metric in [.7,.6,.61,.62,.63]:
    # 일반적인 Epoch를 가정한 Optimizer Update
    opt.zero_grad(set_to_none=True)
    param.square().backward();opt.step()
    plateau.step(metric)
    print(f'Validation={metric:.2f}, next LR={opt.param_groups[0]["lr"]:.4f}')

## 15. AMP 선택 실행

CUDA가 있는 환경에서는 AMP를 사용할 수 있습니다. CPU에서는 일반 Float32 경로로 실행합니다.

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_model=nn.Linear(2,1).to(device)
amp_opt=torch.optim.AdamW(amp_model.parameters(),lr=.01)
xm=x_train[:16].to(device);ym=y_train[:16].to(device)
amp_opt.zero_grad(set_to_none=True)
if device.type=='cuda':
    scaler=torch.amp.GradScaler('cuda')
    with torch.amp.autocast('cuda',dtype=torch.float16):
        amp_loss=nn.MSELoss()(amp_model(xm),ym)
    scaler.scale(amp_loss).backward()
    scaler.unscale_(amp_opt)
    nn.utils.clip_grad_norm_(amp_model.parameters(),1.0)
    scaler.step(amp_opt);scaler.update()
else:
    amp_loss=nn.MSELoss()(amp_model(xm),ym)
    amp_loss.backward()
    nn.utils.clip_grad_norm_(amp_model.parameters(),1.0)
    amp_opt.step()
print('device:',device.type,'loss:',round(float(amp_loss.detach()),4))

## 정리 및 다음 실습

- `train()`, `eval()`, `inference_mode()`는 각각 다른 역할을 합니다.
- Optimizer와 Scheduler의 State를 함께 보존해야 중단 재개가 가능합니다.
- 불완전한 Micro-Batch를 누적할 때는 Sample 비율로 평균 Loss를 보정합니다.
- Early Stopping에서는 최저 Validation Loss의 Weight를 복원합니다.

전체 6개 프로젝트의 Starter Code·Solution·테스트는 Chapter 07의 `projects/`, `solutions/`, `tests/`에 있습니다. 전문 프로젝트는 `projects/professional_01_fleet_training/README.md`를 참고하세요.